# DuckDB CSV Relationship Manager for Google Colab

This notebook creates a DuckDB database, imports CSV files, infers likely relationships, and lets you approve them through a widget-based UI.

In [ ]:
%pip install -q duckdb ipywidgets pandas gspread matplotlib
from google.colab import output, auth
from google.auth import default
output.enable_custom_widget_manager()

try:
    auth.authenticate_user()
    creds, _ = default()
except Exception:
    pass


In [ ]:
from pathlib import Path
import re
import duckdb
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

project_dir = Path('/content/duckdb-csv-gui')
project_dir.mkdir(parents=True, exist_ok=True)
data_dir = project_dir / 'data'
data_dir.mkdir(parents=True, exist_ok=True)


def sanitize_table_name(name):
    cleaned = re.sub(r'[^0-9A-Za-z_]+', '_', name).strip('_')
    if not cleaned:
        return 'table'
    if cleaned[0].isdigit():
        cleaned = f't_{cleaned}'
    return cleaned


def load_csvs(conn, uploaded_files):
    tables = []
    for filename, content_bytes in uploaded_files.items():
        if not filename.lower().endswith('.csv'):
            continue
        csv_path = data_dir / filename
        csv_path.write_bytes(content_bytes)
        table_name = sanitize_table_name(Path(filename).stem)
        conn.execute(f'DROP TABLE IF EXISTS {table_name}')
        conn.execute(f'CREATE TABLE {table_name} AS SELECT * FROM read_csv_auto(?)', [str(csv_path)])
        tables.append(table_name)
    return tables


def infer_relationships(conn, tables):
    proposals = []
    seen = set()
    for index, left_table in enumerate(tables):
        left_cols = [row[1] for row in conn.execute(f"PRAGMA table_info('{left_table}')").fetchall()]
        for right_table in tables[index + 1:]:
            right_cols = [row[1] for row in conn.execute(f"PRAGMA table_info('{right_table}')").fetchall()]
            for left_col in left_cols:
                for right_col in right_cols:
                    if left_col == right_col:
                        reason = 'same column name'
                    elif left_col.endswith('_id') and right_col == 'id':
                        reason = 'child key to primary key'
                    elif left_col == 'id' and right_col.endswith('_id'):
                        reason = 'primary key to child key'
                    else:
                        continue
                    key = (left_table, left_col, right_table, right_col)
                    if key not in seen:
                        seen.add(key)
                        proposals.append({'source_table': left_table, 'source_column': left_col, 'target_table': right_table, 'target_column': right_col, 'reason': reason})
    return proposals


def approve_relationships(conn, selected):
    created = []
    for proposal in selected:
        view_name = sanitize_table_name(f"view_{proposal['source_table']}_{proposal['target_table']}")
        conn.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT a.*, b.* FROM {proposal['source_table']} AS a JOIN {proposal['target_table']} AS b ON a.{proposal['source_column']} = b.{proposal['target_column']}")
        created.append(view_name)
    return created


def get_schema(conn, table_name):
    return conn.execute(f"PRAGMA table_info('{table_name}')").fetchall()


def preview_table(conn, table_name, limit=10):
    return conn.execute(f"SELECT * FROM {table_name} LIMIT {limit}").fetchdf()


def run_query(conn, sql):
    return conn.execute(sql).fetchdf()


def insert_row(conn, table_name, values):
    cols = [row[1] for row in get_schema(conn, table_name)]
    placeholders = ', '.join(['?'] * len(cols))
    col_sql = ', '.join([f'"{c}"' for c in cols])
    conn.execute(f'INSERT INTO {table_name} ({col_sql}) VALUES ({placeholders})', values)


def delete_row(conn, table_name, pk_value):
    pk_col = [row[1] for row in get_schema(conn, table_name) if row[5] == 1][0]
    conn.execute(f'DELETE FROM {table_name} WHERE "{pk_col}" = ?', [pk_value])


def export_to_sheet(df, sheet_name='DuckDB_Report'):
    import gspread
    from google.colab import auth
    auth.authenticate_user()
    gc = gspread.authorize(default()[0])
    sh = gc.create(sheet_name)
    worksheet = sh.sheet1
    worksheet.update([df.columns.values.tolist()] + df.values.tolist())
    return sh.url


def make_chart(df, chart_type='bar'):
    if df.empty:
        raise ValueError('No data to chart.')
    if len(df.columns) < 2:
        raise ValueError('Need at least two columns for a chart.')
    x_col = df.columns[0]
    y_col = df.columns[1]
    fig, ax = plt.subplots(figsize=(6, 4))
    if chart_type == 'bar':
        ax.bar(df[x_col].astype(str), df[y_col])
    elif chart_type == 'line':
        ax.plot(df[x_col].astype(str), df[y_col], marker='o')
    else:
        ax.scatter(df[x_col].astype(str), df[y_col])
    ax.set_title('Chart from query result')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()


db_path = project_dir / 'duckdb_gui.duckdb'
conn = duckdb.connect(str(db_path))

status = widgets.HTML('<b>Status:</b> Ready for your CSV files.')
relation_container = widgets.VBox([])
output_area = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='12px', width='100%'))
checkboxes = []
upload_widget = widgets.FileUpload(accept='.csv', multiple=True)
query_input = widgets.Textarea(value='SELECT * FROM table_name LIMIT 10;', layout=widgets.Layout(width='100%', height='100px'))
table_select = widgets.Dropdown(options=[], description='Table:', layout=widgets.Layout(width='220px'))
chart_type = widgets.Dropdown(options=['bar', 'line', 'scatter'], value='bar', description='Chart:')


def refresh_tables(_=None):
    global checkboxes
    with output_area:
        output_area.clear_output()
        uploaded_files = upload_widget.value or {}
        if not uploaded_files:
            print('No CSV files selected yet.')
            status.value = '<b>Status:</b> Waiting for CSV upload.'
            table_select.options = []
            return
        tables = load_csvs(conn, uploaded_files)
        table_select.options = tables
        proposals = infer_relationships(conn, tables)
        checkboxes = []
        if not proposals:
            relation_container.children = [widgets.HTML('No obvious relationships were detected. You can still query the tables directly.')]
            status.value = f'<b>Status:</b> Loaded {len(tables)} table(s).'
            return
        rows = []
        for proposal in proposals:
            cb = widgets.Checkbox(value=True, description=f"{proposal['source_table']}.{proposal['source_column']} -> {proposal['target_table']}.{proposal['target_column']}")
            cb.proposal = proposal
            checkboxes.append(cb)
            rows.append(widgets.HBox([cb, widgets.Label(proposal['reason'])]))
        relation_container.children = rows
        status.value = f'<b>Status:</b> Loaded {len(tables)} table(s) and inferred relationships.'


def approve(_=None):
    with output_area:
        output_area.clear_output()
        selected = [cb.proposal for cb in checkboxes if cb.value]
        if not selected:
            print('No relationships selected.')
            return
        created = approve_relationships(conn, selected)
        print(f'Approved {len(selected)} relationship(s). Views created: {created}')


def show_table(_=None):
    with output_area:
        output_area.clear_output()
        if not table_select.value:
            print('Select a table first.')
            return
        display(Markdown(f'### {table_select.value}'))
        display(preview_table(conn, table_select.value))
        display(Markdown('#### Schema'))
        display(pd.DataFrame(get_schema(conn, table_select.value), columns=['cid','name','type','notnull','dflt_value','pk']))


def run_sql(_=None):
    with output_area:
        output_area.clear_output()
        try:
            result = run_query(conn, query_input.value)
            display(result)
        except Exception as e:
            print(f'Error: {e}')


def run_sql_and_export(_=None):
    with output_area:
        output_area.clear_output()
        try:
            result = run_query(conn, query_input.value)
            display(result)
            sheet_url = export_to_sheet(result, sheet_name='DuckDB_Report')
            print(f'Exported to Google Sheets: {sheet_url}')
        except Exception as e:
            print(f'Error: {e}')


def run_sql_and_chart(_=None):
    with output_area:
        output_area.clear_output()
        try:
            result = run_query(conn, query_input.value)
            display(result)
            make_chart(result, chart_type=chart_type.value)
        except Exception as e:
            print(f'Error: {e}')


def add_row(_=None):
    with output_area:
        output_area.clear_output()
        if not table_select.value:
            print('Select a table first.')
            return
        schema = get_schema(conn, table_select.value)
        cols = [row[1] for row in schema]
        print('Enter values separated by commas:')
        print(', '.join(cols))
        values = input().split(',')
        cleaned = [v.strip() for v in values]
        insert_row(conn, table_select.value, cleaned)
        print('Row inserted successfully.')


def remove_row(_=None):
    with output_area:
        output_area.clear_output()
        if not table_select.value:
            print('Select a table first.')
            return
        pk_value = input('Enter the primary key value to delete: ')
        delete_row(conn, table_select.value, pk_value)
        print('Row deleted successfully.')

load_button = widgets.Button(description='Load Uploaded CSVs', button_style='primary')
approve_button = widgets.Button(description='Approve Selected', button_style='success')
show_button = widgets.Button(description='Show Table')
query_button = widgets.Button(description='Run SQL')
export_button = widgets.Button(description='Run SQL + Export to Sheets')
chart_button = widgets.Button(description='Run SQL + Chart')
insert_button = widgets.Button(description='Insert Row')
delete_button = widgets.Button(description='Delete Row')

load_button.on_click(refresh_tables)
approve_button.on_click(approve)
show_button.on_click(show_table)
query_button.on_click(run_sql)
export_button.on_click(run_sql_and_export)
chart_button.on_click(run_sql_and_chart)
insert_button.on_click(add_row)
delete_button.on_click(remove_row)

header = widgets.HTML('<div style="font-size: 18px; font-weight: 600; color: #1f2937;">DuckDB CSV Relationship Manager</div>')
subheader = widgets.HTML('Upload CSV files, inspect tables, run SQL, approve relationships, and visualize results in a polished, boss-friendly workflow.')

ui = widgets.VBox([
    header,
    subheader,
    widgets.HTML('<b>1. Upload CSV files</b>'),
    upload_widget,
    widgets.HBox([load_button, approve_button]),
    widgets.HTML('<b>2. Inspect and query</b>'),
    widgets.HBox([table_select, show_button, insert_button, delete_button]),
    query_input,
    widgets.HBox([query_button, export_button, chart_button, chart_type]),
    status,
    relation_container,
    output_area,
], layout=widgets.Layout(width='100%'))

display(ui)
